In [20]:
%pip install sentence-transformers
%pip install -qU langchain-huggingface

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from pathlib import Path
import numpy as np
from langchain_openai import OpenAIEmbeddings


e:\graduation_project\gradproj_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# -----------------------------
# 1️⃣ List available JSON files
json_dir = Path(r"E:/graduation_project/json_llm_responses")
print(json_dir.exists())  # Should print True

json_files = list(json_dir.glob("*.json"))
print("Available JSON files:")
for i, f in enumerate(json_files, 1):
    print(f"{i}. {f.name}")



True
Available JSON files:
1. refrence_1_after_llm_preprocessing.json


In [4]:
# -----------------------------
# 2️⃣ Choose a file
choice = int(input("Select JSON file number to generate embeddings: "))
selected_file = json_files[choice - 1]
print(f"Selected: {selected_file.name}")

Selected: refrence_1_after_llm_preprocessing.json


In [6]:
# -----------------------------
# 3️⃣ Load chunks
with open(selected_file, "r", encoding="utf-8") as f:
    chunks = json.load(f)

In [7]:
from langchain_core.documents import Document

In [8]:
# 2. Convert to LangChain 'Document' Format
documents = []
kept_count = 0
for chunk in chunks:
    # SKIP logic: If keep is False, ignore it
    if chunk.get("keep") is False:
        continue
        
    # CONSTRUCT CONTENT: This is what the AI "reads"
    # We combine Metadata + Text so the AI understands context
    # Example: "Chapter 1 > Introduction: A computer system consists of..."
    topic_str = f"{chunk.get('topic', 'General')} > {chunk.get('subtopic', '')}"
    page_content = f"{topic_str}\n{chunk['clean_text']}"
    
    # METADATA: Useful for filtering later (e.g., "Search only Chapter 1")
    metadata = {
        "id": chunk["id"],
        "topic": chunk.get("topic", "General"),
        "subtopic": chunk.get("subtopic", ""),
        "source": chunk.get("source", "Textbook") # Default if source is missing
    }
    
    # Create Document Object
    doc = Document(page_content=page_content, metadata=metadata)
    documents.append(doc)
    kept_count += 1

print(f"✅ Filtered down to {kept_count} useful documents.")

✅ Filtered down to 2910 useful documents.


In [8]:
%pip install -qU "langchain-chroma>=0.1.2"


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import shutil
from langchain_chroma import Chroma
# 1. SETUP PATH



# Output: The specific directory you asked for
chroma_dir = Path("../vector_store/chroma_db")

# Optional: Clear old DB if you want a fresh start
#if chroma_dir.exists():
#    shutil.rmtree(chroma_dir)

In [10]:
# -----------------------------
# 5️⃣ Initialize embeddings model
from langchain_huggingface import HuggingFaceEmbeddings
from openai import embeddings

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print(f"💾 Creating ChromaDB at: {chroma_dir.resolve()} (Stage 3)...")
vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory=str(chroma_dir) # Must be string for Chroma
)


💾 Creating ChromaDB at: E:\graduation_project\vector_store\chroma_db (Stage 3)...


In [11]:
print("-" * 30)
print(f"🎉 SUCCESS! Database ready at: {chroma_dir}")
print(f"Total Vectors Stored: {vector_db._collection.count()}")

------------------------------
🎉 SUCCESS! Database ready at: ..\vector_store\chroma_db
Total Vectors Stored: 2910
